## Embeddings in Deep Learning
---

*What are Embeddings?*

In deep learning, **embeddings** are dense vector representations of data. They are essentially learned numerical representations of input data (e.g., images, words, sentences) in a continuous vector space. Embeddings are particularly powerful because they encode **meaningful features** that capture the intrinsic relationships within the data.

For Convolutional Neural Networks (CNNs), embeddings are typically found in the final layer (or penultimate layer, the last layer *before* the output layer) of the network before classification. 

At this stage, the CNN has distilled the input image into a highly compressed form—a vector containing high-level abstract features learned during training.

---

*Why are Embeddings important?*

Embeddings are essential in deep learning because they:

1. **Capture High-Level Features:** The embeddings represent a distilled version of the most relevant information in the input (e.g., visual features in images).
2. **Enable Comparisons:** The embeddings of similar inputs are close in vector space, enabling us to compute similarity or clustering.
3. **Transfer Knowledge:** Pre-trained CNN embeddings can generalize well to new tasks without requiring re-training, saving computational resources.
4. **Reduce Dimensionality:** Raw input data (e.g., pixels of an image) is typically very high-dimensional. Embeddings reduce this to a compact vector representation while retaining meaningful features.

---


*How CNNs work (a brief recap)*

CNNs process images through a series of **convolutions, pooling, and nonlinear activations** to extract increasingly abstract features:

- **Early layers** focus on low-level features like edges, textures, or colors.
- **Deeper layers** learn higher-level features, such as shapes, objects, or patterns.
- **Output layer** delivers a task tailerd output, e.g., class scores (for classification) or a continous number (for regression).

---

*Where Embeddings fit in CNNs*

Usually, the **last hidden layer** (the output layer) is where embeddings are most commonly extracted from. This layer represents the high-level features learned by the CNN in a compact vector. For example:

- In a CNN trained on ImageNet, the last layer might output a **1000-dimensional vector** for classification across 1000 categories.
- By removing the classification layer, the model's last hidden layer instead outputs a **feature vector (embedding)** for any input image, which can be used for tasks like similarity search, clustering, and transfer learning.

---

## Key Applications of CNN Embeddings

**1. Image Similarity Search**

CNN embeddings allow us to compare images by computing the **distance** (e.g., cosine similarity or Euclidean distance) between their feature vectors. For instance:

- Images of dogs will have embeddings that are closer together in vector space than embeddings of dogs and cats.
- This makes embeddings ideal for **content-based image retrieval (CBIR)** systems.

**2. Transfer Learning**

CNN embeddings allow us to transfer knowledge from a pre-trained model to a new task. Instead of training a new model from scratch:

- Use a pre-trained CNN to process all the images in your dataset in order to generate rich embeddings for each one of them. 
- Use these embeddings as input features to our new model.
- Train a lightweight classifier or regressor on these features for your task.


**3. Clustering and Dimensionality Reduction**

Embedding spaces make it easier to cluster images into groups or visualize them using techniques like **t-SNE** or **UMAP**.


**4. Anomaly Detection**

If the embeddings of a CNN are clustered around typical examples in your dataset, any input image that generates an **outlier embedding** (far from the cluster) can be flagged as anomalous.

---

##  How do Embeddings work?

Some key properties of Embeddings are as follows

1. **Proximity Represents Similarity:** Similar inputs produce embeddings that are close in vector space.
2. **Disentangled Features:** Embeddings separate distinct features (e.g., color, texture, shape) into independent dimensions.
3. **Generalization:** Pre-trained embeddings generalize well across domains.

---

## Code Example - Extracting and using Image Embeddings

We’ll use PyTorch to load a pre-trained ResNet-50 model and remove its final classification layer. We'll then use the model to generate embeddings for the images we feed into it.

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Load Pre-trained ResNet-50
model = models.resnet50(pretrained=True)

# Inspect the model architecture
model

/Users/wijdancederlid/Desktop/deep_ML/deeplearning/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/wijdancederlid/Desktop/deep_ML/deeplearning/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/wijdancederlid/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [2]:
list(model.children())[:-1]

[Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
 BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True),
 MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False),
 Sequential(
   (0): Bottleneck(
     (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (downsample): Sequential(
       (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
       (1): BatchNorm2d(256, eps=1e-05, momentum

In [3]:
# Remove the classification head

model = nn.Sequential(*list(model.children())[:-1])


In [4]:
# Inspect the model
# Note that this will now output a 2048-dimensional vector, for each input image

model

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [5]:
# Define a tranformation to preprocess the input image, to the expected 224x224 input size of the ResNet-50 model

transform = transforms.Compose([
                                transforms.Resize((224, 224)),  # Resize to 224x224 (ResNet input size)
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize as ResNet expects
                               ])


# Define a function to get the embedding for an image

def get_embedding(image_path, model):
    
    model.eval()
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)  # Add batch dimension

    # Generate embedding
    with torch.no_grad():
        embedding = model(image).squeeze()  # Remove batch and spatial dimensions

    result = embedding.numpy().reshape(1, -1) # Convert to numpy array and reshape to (1, output_dim_size)
 
    return result

---

## Excercises

### Problem 1.

**a)**

Begin by downloading (from arbitrary source) images of cats, dogs and birds. Make sure to have 3 different images of each animal.

Then, generate and extract embeddings for all 9 images. 

Once that's done, start calculating the cosine similarity (imported from sklearn, above) between each pair of embeddings for your animal pictures. Does the results make sense?

*Also, do you recall cosine similarity from Linear Algebra course? :)*

If not, here's the official documentation https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

What does the cosine_similarity measure, do you think?



In [19]:
print(get_embedding('images/dog1.jpeg', model))
print(get_embedding('images/dog2.jpeg', model))
print(get_embedding('images/dog3.jpeg', model))
print(get_embedding('images/bird1.jpeg', model))
print(get_embedding('images/bird2.jpeg', model))
print(get_embedding('images/bird3.jpeg', model))
print(get_embedding('images/cats1.jpeg', model))
print(get_embedding('images/cats2.jpeg', model))
print(get_embedding('images/cats3.jpeg', model))

[[0.33030933 0.11601384 0.32027245 ... 0.09483325 0.52507865 0.12743469]]
[[0.7193672  0.34096676 0.84376484 ... 0.13982451 0.12345359 0.1522871 ]]
[[0.9453175  0.28121093 0.2765915  ... 0.21116112 0.4430702  0.4174455 ]]
[[0.17517431 0.15743206 0.01737427 ... 0.2659821  0.44892332 0.35657033]]
[[0.9188128  0.5965008  0.3372234  ... 0.29275617 0.18848628 0.83676153]]
[[0.2561109  0.33095002 0.19312216 ... 0.3045358  0.4719611  0.24047144]]
[[0.6275722  0.45532963 0.10133222 ... 0.48299074 0.48845002 0.30445394]]
[[0.5719677  0.07194386 0.0209606  ... 0.03121912 0.13688067 0.1346602 ]]
[[0.24820304 0.04722444 0.01477583 ... 0.23993054 0.37808308 0.5362047 ]]


In [25]:
print(get_embedding('images/cats3.jpeg', model).shape)  # Should print (1, 2048)


(1, 2048)


In [21]:
# Example paths - adjust these to your actual file names!
image_paths = [
    'images/cats1.jpeg', 'images/cats2.jpeg', 'images/cats3.jpeg',
    'images/dog1.jpeg', 'images/dog2.jpeg', 'images/dog3.jpeg',
    'images/bird1.jpeg', 'images/bird2.jpeg', 'images/bird3.jpeg'
]

embeddings = []
for path in image_paths:
    emb = get_embedding(path, model)
    embeddings.append(emb)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embeddings_array = np.vstack(embeddings)

sim_matrix = cosine_similarity(embeddings_array)

print(sim_matrix)

[[1.0000001  0.7936015  0.75372225 0.63219213 0.6997006  0.67908055
  0.5927041  0.627622   0.62576205]
 [0.7936015  1.         0.7504086  0.49475545 0.58601683 0.5564937
  0.48174638 0.5012041  0.50419   ]
 [0.75372225 0.7504086  1.0000002  0.5285008  0.53787726 0.54457057
  0.5372639  0.5480355  0.53430855]
 [0.63219213 0.49475545 0.5285008  1.0000001  0.68546474 0.7304595
  0.6635918  0.6137399  0.6428435 ]
 [0.6997006  0.58601683 0.53787726 0.68546474 1.0000005  0.7260254
  0.58135766 0.581361   0.60157144]
 [0.67908055 0.5564937  0.54457057 0.7304595  0.7260254  1.0000004
  0.6600882  0.62357754 0.6963495 ]
 [0.5927041  0.48174638 0.5372639  0.6635918  0.58135766 0.6600882
  0.9999998  0.71604174 0.80084604]
 [0.627622   0.5012041  0.5480355  0.6137399  0.581361   0.62357754
  0.71604174 0.99999994 0.76112616]
 [0.62576205 0.50419    0.53430855 0.6428435  0.60157144 0.6963495
  0.80084604 0.76112616 1.        ]]


**b)**

Repeat the above excercise but now instead of cosine similarity, calculate the *euclidean distance* between the all the image embeddings pairs. Does the result make sense?

*Note: what does the euclidean distance (pythagoran distance) between two vectors mean?*

In [23]:
from sklearn.metrics.pairwise import euclidean_distances

dist_matrix = euclidean_distances(embeddings_array)

print("Euclidean Distance Matrix:")
print(dist_matrix)

Euclidean Distance Matrix:
[[ 0.       16.312206 18.624918 21.132166 19.549988 20.697412 22.312572
  22.086718 20.508057]
 [16.312206  0.       18.80274  24.871084 23.053482 24.417456 25.279066
  25.664095 23.63738 ]
 [18.624918 18.80274   0.       25.143595 25.38628  25.69563  24.986238
  25.405972 24.190933]
 [21.132166 24.871084 25.143595  0.       19.50058  18.583275 19.696812
  21.959538 19.301542]
 [19.549988 23.053482 25.38628  19.50058   0.       19.093903 22.567436
  23.371088 21.075863]
 [20.697412 24.417456 25.69563  18.583275 19.093903  0.       20.894226
  22.660295 19.155201]
 [22.312572 25.279066 24.986238 19.696812 22.567436 20.894226  0.
  18.91254  14.597523]
 [22.086718 25.664095 25.405972 21.959538 23.371088 22.660295 18.91254
   0.       16.883831]
 [20.508057 23.63738  24.190933 19.301542 21.075863 19.155201 14.597523
  16.883831  0.      ]]


**c)**

Can we draw any conclusions from the above results? What does it mean for the embeddings of two images to be close or far apart? Are the results "perfect"?

**d)**

Now try another pre-trained model and repeat the above. Do you get similar, better or worse results? What conclusions can you draw?

### Problem 2

Use-cases everywhere!

Look at the list of example use-cases above. Can you think of concrete examples of each? Write down specific applications for each use-case, and discuss with your classmates.

### Problem 3

Tiiituuut - Vem där?! :)

Let's apply what we've learned so far to try building a *face recognition* app. 

**a)**

Start by collecting a bunch of pictures of yourself, and then some of your friends and/or random people you find on the internet. 

Then, repeat the excercise from Problem 1, but now with these images instead. Can you accurately pinpoint yourself in images?


.



**b)**

Ok, let's get serious with a challange here. Assume now that we want to create an app that connect to your webcam (if you have one) and analyze pictures taken from it. If you don't have a webcam, team up with a friend that does.

This problem can be split into a few steps:

1. Connect to your webcam via Python. Search online on how to do it.
2. Take pictures in regular intervals (e.g. every 5 seconds, or perhaps less).
3. Generate embeddings of these images the moment they are taken, and compare them with the embeddings of the images you've collected in **a)**. If the embeddings are close, you've hopefully found a match!
4. Display the result on the screen somehow. 

If done correctly, this will work as "real-time" face recognition app!

*Note: How will you compare the embeddings of the images taken from the camera, with the existing images of yourself? What criteria do you set for a match? Discuss with your classmates.*